# Voice v0: a spoken shop call (ears → brain → mouth), free models on Kaggle's free GPU

**What it does:** the AI asks a shopkeeper 3 questions in Hindi, by voice (in stock? price? delivery?).
Each spoken answer goes through three free models, one after the other:
1. **Ears**: Whisper large-v3 turbo turns the speech into text (Hindi).
2. **Brain**: the trained shop model (Ollama, the repo's `Modelfile`) reads the text and gives the 4 fields
   (in stock, price, delivery, other product offered). A small template writes a short polite reply in Hindi.
3. **Mouth**: a text-to-speech model says the reply.

It times each step and saves everything to the **Output** tab, folder `voice/`: the questions (`q1.wav`–`q3.wav`),
the test answers (`synthetic_1.wav`…), the replies (`reply_1.wav`…) and `voice_results.json`.

**Test answers:** your own recordings if a Kaggle dataset with them is attached (`q2_anything.m4a` answers
question 2), and always 6 made-up answers spoken by the TTS voice. Those check the plumbing, not real accents.

**Settings (right side):** Input `wimb-shop-model-training` (the trained model), Accelerator **GPU T4**,
Internet **on**. For the better Hindi voice (Indic Parler-TTS) add a Hugging Face token as the secret `HF_TOKEN`
(Add-ons → Secrets); without it a fallback voice is used (MMS-TTS, non-commercial licence: test only).

**Or with no clicks:** on GitHub, **Actions → Voice v0 (speech to speech) → Run workflow**, or ask Claude.

In [ ]:
#@title 1. Settings: the product, the models, your recordings
import json, os, re, shutil, subprocess, sys, time, urllib.error, urllib.request

PRODUCT = "16A MCB"  # what the AI asks about (and what the shop model is told)
PRODUCT_SPOKEN = "सोलह ए एमसीबी"  # the same, as the voice says it (Devanagari: some voices skip Latin letters)
ASR_MODEL = "openai/whisper-large-v3-turbo"  # speech to text (MIT licence, not gated)
# Text to speech, tried in this order: the first that works is used (its licence is saved with the results).
TTS_CANDIDATES = ["ai4bharat/indic-parler-tts", "facebook/mms-tts-hin"]
CLIPS_DIR = "/kaggle/input"  # your recordings: any .wav/.mp3/.m4a/.ogg/.opus under here (an attached Kaggle dataset)
MAX_CLIPS = 20  # at most this many of your recordings
WORKING_DIR = "/kaggle/working/voice"  # everything is saved here (the notebook's Output tab)

In [ ]:
#@title 2. Helpers (no need to read; built from score.py, voice/voicecall.py and Modelfile)
"""Shared by the notebook and the tests: the task's prompt, and how an answer is scored.

The task: read what a shopkeeper said on a phone call and return JSON with four fields.
"""

import json
import re
import time

SYSTEM = (
    "You read what a shopkeeper said on a phone call about a product. "
    "Reply with only a JSON object with these keys: "
    "in_stock (true, false or null), price_inr (a number or null), "
    "delivery (true, false or null), alternative (text or null). "
    "price_inr is the price of the product that was asked for. "
    "Use null when the shopkeeper did not say."
)

FIELDS = ("in_stock", "price_inr", "delivery", "alternative")


def user_message(product: str, said: str) -> str:
    return f"Product: {product}\nShopkeeper: {said}"


def answer_text(answer: dict) -> str:
    """The exact text the model is trained to reply with."""
    return json.dumps({k: answer[k] for k in FIELDS}, ensure_ascii=False)


def parse(text: str) -> dict | None:
    """The last JSON object in the reply that has exactly the four keys, or None.

    Every "{" is tried, so braces inside a value ("M-seal {2 pack}") or other text around the
    answer do not hide it.
    """
    decoder = json.JSONDecoder()
    found = None
    for match in re.finditer(r"\{", text):
        try:
            value, _ = decoder.raw_decode(text, match.start())
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict) and set(value) == set(FIELDS):
            found = value
    return found


def _tokens(text: str) -> set[str]:
    """Lower-case words, with "40mm" read as "40 mm" and punctuation dropped."""
    text = re.sub(r"(?<=\d)(?=[a-z])|(?<=[a-z])(?=\d)", " ", text.lower())
    return set(re.sub(r"[-_.,;:!?()\[\]{}\"'/]", " ", text).split())


def same(field: str, got, want) -> bool:
    if want is None or got is None:
        return got is None and want is None
    if field in ("in_stock", "delivery"):
        return isinstance(got, bool) and got == want  # 1 and 0 are not true and false
    if field == "price_inr":
        if isinstance(got, bool):
            return False
        try:
            return float(got) == float(want)
        except (TypeError, ValueError):
            return False
    # alternative: the wanted words must all be there, with at most two extra ("40 mm padlock").
    if not isinstance(got, str):
        return False
    have, need = _tokens(got), _tokens(want)
    return bool(have) and need <= have and len(have - need) <= 2


def score(replies: list[str], examples: list[dict]) -> dict:
    """Per-field accuracy, all-four-right accuracy, and the mistakes.

    A reply with no valid answer object (or a missing or extra key) gets every field wrong.
    """
    right = {f: 0 for f in FIELDS}
    all_right = json_ok = 0
    mistakes = []
    for reply, ex in zip(replies, examples, strict=True):
        got = parse(reply)
        if got is not None:
            json_ok += 1
        wrong = list(FIELDS) if got is None else [f for f in FIELDS if not same(f, got[f], ex["answer"][f])]
        for f in FIELDS:
            right[f] += f not in wrong
        if wrong:
            mistakes.append({"said": ex["said"], "want": ex["answer"], "reply": reply, "wrong": wrong})
        else:
            all_right += 1
    n = len(examples)
    return {
        "examples": n,
        "json_ok": json_ok / n,
        "all_correct": all_right / n,
        **{f: right[f] / n for f in FIELDS},
        "mistakes": mistakes,
    }


KEYS = ("all_correct", "json_ok", *FIELDS)


def evaluate(ask, sets: dict[str, list[dict]], deadline: float | None = None) -> dict:
    """Run ask(product, said) over every test set and score it. Every reply is kept.

    After `deadline` (a time.time() value) no more questions are asked; what was answered is
    scored, and "of" says how many there were.
    """
    out = {}
    for name, rows in sets.items():
        replies = []
        for ex in rows:
            if deadline is not None and time.time() > deadline:
                break
            replies.append(ask(ex["product"], ex["said"]))
        done = rows[: len(replies)]
        scored = score(replies, done) if done else {"examples": 0, "mistakes": []}
        out[name] = {**scored, "replies": replies, "of": len(rows)}
    return out


def show(label: str, results: dict, max_mistakes: int = 8) -> None:
    for name, res in results.items():
        n = res["examples"]
        if not n:
            print(f"== {label} [{name}]: no calls answered")
            continue
        partial = f" (only {n} of {res['of']} answered)" if n < res["of"] else ""
        print(f"== {label} [{name}]: all 4 fields right in {res['all_correct']:.0%} of {n} calls{partial}")
        print("   " + "   ".join(f"{k} {res[k]:.0%}" for k in KEYS[1:]))
        for m in res["mistakes"][:max_mistakes]:
            print("\n   said: ", m["said"])
            print("   want: ", answer_text(m["want"]))
            print("   reply:", m["reply"][:200])
            print("   wrong:", ", ".join(m["wrong"]))


def results_payload(model: str, epochs: int, before: dict, after: dict) -> dict:
    """What results.json holds: scores, mistakes and every raw reply, so scores can be recomputed."""
    sets = {}
    for name, a in after.items():
        b = before[name]
        sets[name] = {
            "examples": a["of"],
            "before": {k: round(b[k], 3) for k in KEYS},
            "after": {k: round(a[k], 3) for k in KEYS},
            "mistakes_before": b["mistakes"],
            "mistakes_after": a["mistakes"],
            "replies_before": b["replies"],
            "replies_after": a["replies"],
        }
    return {"model": model, "epochs": epochs, "sets": sets}


def make_modelfile(gguf_name: str) -> str:
    """An Ollama Modelfile whose template is exactly the text the model was trained on.

    Unsloth's own Qwen3 template differs a little (a blank line after the system message, and
    thinking-mode syntax that needs a newer Ollama), so it is not used.
    """
    return (
        f"FROM ./{gguf_name}\n"
        'TEMPLATE """{{ if .System }}<|im_start|>system\n{{ .System }}<|im_end|>\n{{ end }}'
        '{{ if .Prompt }}<|im_start|>user\n{{ .Prompt }}<|im_end|>\n{{ end }}'
        '<|im_start|>assistant\n{{ .Response }}<|im_end|>\n"""\n'
        'PARAMETER stop "<|im_end|>"\n'
        "PARAMETER temperature 0\n"
        # Calls are short (training used 1,024 tokens). Ollama's default context (32,768 on 0.40) takes about
        # 7.5 GB of memory with this model; 2,048 needs about 3 GB, so it fits a laptop with 8 GB.
        "PARAMETER num_ctx 2048\n"
        f'SYSTEM """{SYSTEM}"""\n'
    )


def in_words(answer):
    """An answer in plain words, e.g. "in stock: yes; price: ₹220; delivery: no; other product offered: none"."""
    if not isinstance(answer, dict):
        return "no answer (the reply was not the 4-field JSON)"

    def yes_no(value):
        if value is None:
            return "not said"
        return ("yes" if value else "no") if isinstance(value, bool) else repr(value)

    price = answer.get("price_inr")
    if price is None:
        price = "not said"
    elif isinstance(price, (int, float)) and not isinstance(price, bool):
        price = f"₹{int(price)}" if float(price).is_integer() else f"₹{price}"
    else:
        price = repr(price)
    alternative = answer.get("alternative")
    return (f"in stock: {yes_no(answer.get('in_stock'))}; price: {price}; delivery: {yes_no(answer.get('delivery'))}; "
            f"other product offered: {'none' if alternative is None else alternative}")


"""Voice v0: the plain parts of the spoken shop call, shared by voice_v0.ipynb (built in by voice/build_voice.py)
and the tests. No torch here: the ears (ASR), the mouth (TTS) and the brain (the shop model in Ollama) are passed in
as plain functions, so the tests can replace them.

    transcribe(path, language) -> text       language "hi", or None to let Whisper guess
    think(product, transcript) -> reply       the shop model's raw reply (the 4-field JSON)
    speak(text, path, voice) -> None          writes a .wav; voice "ai" or "shopkeeper"
"""

import os
import re
import time


RESULTS_FILE = "voice_results.json"
AUDIO = (".wav", ".mp3", ".m4a", ".ogg", ".opus")  # .opus: WhatsApp voice notes

# Every model the cascade can use, with its licence. business_use False: allowed for this internal test only.
ASR_LICENCES = {"openai/whisper-large-v3-turbo": "MIT"}
TTS_INFO = {
    "ai4bharat/indic-parler-tts": {"licence": "Apache-2.0", "needs_token": True, "business_use": True,
                                   "note": "needs a Hugging Face token whose account accepted the model's terms"},
    "facebook/mms-tts-hin": {"licence": "CC-BY-NC-4.0", "needs_token": False, "business_use": False,
                             "note": "non-commercial licence: allowed for this internal test only, not for business use"},
}
BRAIN_INFO = "wimb-shop: Qwen3-4B-Instruct-2507 (Apache-2.0) + our LoRA, Q4_K_M in Ollama (MIT)"

# What the AI asks: (what is shown, what is said). {p} is the product as written, {s} as spoken.
QUESTIONS = [
    ("नमस्ते, क्या आपके पास {p} है?", "नमस्ते, क्या आपके पास {s} है?"),
    ("{p} का दाम कितना है?", "{s} का दाम कितना है?"),
    ("क्या आप घर पर डिलीवरी करते हैं?", "क्या आप घर पर डिलीवरी करते हैं?"),
]

# Made-up shopkeeper answers about a 16A MCB, spoken by the TTS so the pipeline runs end to end without real clips.
# They are a check of the plumbing, not of real accents. Devanagari only: digits and Latin letters can't be spoken by
# every TTS. question: which question it answers (None: none). alternative_any: any of these words counts as the
# alternative (the model may write it in Latin or in Devanagari).
SYNTHETIC = [
    {"question": 1, "said": "हाँ जी, सोलह ए वाला एमसीबी है। दो सौ बीस रुपये का पड़ेगा।",
     "answer": {"in_stock": True, "price_inr": 220, "delivery": None, "alternative": None}},
    {"question": 1, "said": "नहीं भाई, वो वाला अभी खत्म है। हैवेल्स का बत्तीस ए वाला रखा है, वो ले लो।",
     "answer": {"in_stock": False, "price_inr": None, "delivery": None, "alternative": "Havells 32A"},
     "alternative_any": ["havells", "हैवेल्स"]},
    {"question": 3, "said": "हाँ है, ढाई सौ का है, और घर पे भी पहुँचा देंगे।",
     "answer": {"in_stock": True, "price_inr": 250, "delivery": True, "alternative": None}},
    {"question": 3, "said": "है, एक सौ नब्बे का। पर डिलीवरी नहीं करते, दुकान से ले जाना पड़ेगा।",
     "answer": {"in_stock": True, "price_inr": 190, "delivery": False, "alternative": None}},
    {"question": 2, "said": "यस, स्टॉक में है। रेट टू थर्टी है, होम डिलीवरी कल कर देंगे।",
     "answer": {"in_stock": True, "price_inr": 230, "delivery": True, "alternative": None}},
    {"question": None, "said": "पता नहीं, बाद में कॉल करो।",
     "answer": {"in_stock": None, "price_inr": None, "delivery": None, "alternative": None}},
]

ONES = [  # 0 to 99 in Hindi words
    "शून्य", "एक", "दो", "तीन", "चार", "पाँच", "छह", "सात", "आठ", "नौ", "दस", "ग्यारह", "बारह", "तेरह", "चौदह",
    "पंद्रह", "सोलह", "सत्रह", "अठारह", "उन्नीस", "बीस", "इक्कीस", "बाईस", "तेईस", "चौबीस", "पच्चीस", "छब्बीस",
    "सत्ताईस", "अट्ठाईस", "उनतीस", "तीस", "इकतीस", "बत्तीस", "तैंतीस", "चौंतीस", "पैंतीस", "छत्तीस", "सैंतीस",
    "अड़तीस", "उनतालीस", "चालीस", "इकतालीस", "बयालीस", "तैंतालीस", "चवालीस", "पैंतालीस", "छियालीस", "सैंतालीस",
    "अड़तालीस", "उनचास", "पचास", "इक्यावन", "बावन", "तिरपन", "चौवन", "पचपन", "छप्पन", "सत्तावन", "अट्ठावन", "उनसठ",
    "साठ", "इकसठ", "बासठ", "तिरसठ", "चौंसठ", "पैंसठ", "छियासठ", "सड़सठ", "अड़सठ", "उनहत्तर", "सत्तर", "इकहत्तर",
    "बहत्तर", "तिहत्तर", "चौहत्तर", "पचहत्तर", "छिहत्तर", "सतहत्तर", "अठहत्तर", "उनासी", "अस्सी", "इक्यासी", "बयासी",
    "तिरासी", "चौरासी", "पचासी", "छियासी", "सत्तासी", "अट्ठासी", "नवासी", "नब्बे", "इक्यानवे", "बानवे", "तिरानवे",
    "चौरानवे", "पचानवे", "छियानवे", "सत्तानवे", "अट्ठानवे", "निन्यानवे",
]


def hindi_number(n: int) -> str:
    """A whole number in Hindi words, e.g. 220 -> "दो सौ बीस", 1500 -> "एक हज़ार पाँच सौ". Digits from a crore up."""
    if n < 0 or n >= 10_000_000:
        return str(n)
    if n < 100:
        return ONES[n]
    words = []
    for size, name in ((100_000, "लाख"), (1000, "हज़ार"), (100, "सौ")):
        if n >= size:
            words.append(f"{ONES[n // size]} {name}")
            n %= size
    if n:
        words.append(ONES[n])
    return " ".join(words)


def rupees(price) -> tuple[str, str]:
    """A price as (shown, spoken): 220 -> ("₹220", "दो सौ बीस रुपये")."""
    value = float(price)
    whole, paise = int(value), round((value - int(value)) * 100)
    shown = f"₹{whole}" if not paise else f"₹{value:.2f}"
    spoken = f"{hindi_number(whole)} रुपये" + (f" {hindi_number(paise)} पैसे" if paise else "")
    return shown, spoken


def questions(product: str, spoken: str | None = None) -> list[dict]:
    """The 3 questions about this product: [{"n", "text" (shown), "speech" (what the TTS says)}]."""
    return [{"n": n, "text": text.format(p=product), "speech": speech.format(s=spoken or product)}
            for n, (text, speech) in enumerate(QUESTIONS, 1)]


def is_price(value) -> bool:
    return isinstance(value, (int, float)) and not isinstance(value, bool) and value >= 0


def write_reply(answer, product: str, spoken: str | None = None) -> dict:
    """A short polite Hindi reply to what the shop model understood: {"text" (shown), "speech" (for the TTS)}.

    For example: in stock, ₹220, no delivery -> "ठीक है, 16A MCB ₹220 में है, डिलीवरी नहीं। धन्यवाद!".
    Nothing understood (no answer, or all four fields null) -> a polite request to say it again.
    """
    answer = answer if isinstance(answer, dict) else {}
    in_stock, price, delivery, alternative = (answer.get(f) for f in FIELDS)
    price = price if is_price(price) else None
    alternative = alternative.strip() if isinstance(alternative, str) and alternative.strip() else None
    if in_stock is None and price is None and delivery is None and alternative is None:
        sorry = "माफ़ कीजिए, ठीक से समझ नहीं आया। क्या आप एक बार फिर बता सकते हैं? नहीं तो हम बाद में कॉल कर लेंगे। धन्यवाद!"
        return {"text": sorry, "speech": sorry}
    out = {}
    for key, name, money in (("text", product, 0), ("speech", spoken or product, 1)):
        cost = rupees(price)[money] if price is not None else None
        clauses = []
        if in_stock is True:
            clauses.append(f"{name} {cost} में है" if cost else f"{name} है")
        elif in_stock is False:
            clauses.append(f"{name} अभी नहीं है")
        elif cost:
            clauses.append(f"{name} का दाम {cost} है")
        if delivery is True:
            clauses.append("डिलीवरी हो जाएगी")
        elif delivery is False:
            clauses.append("डिलीवरी नहीं")
        text = "ठीक है, " + ", ".join(clauses) + "।" if clauses else "ठीक है।"
        if alternative:
            spoken_digits = re.sub(r"\d+", lambda m: f" {hindi_number(int(m[0]))} ", alternative)  # 32A -> बत्तीस A
            other = alternative if key == "text" else " ".join(spoken_digits.split())
            text += f" {other} के बारे में बताया, हम ग्राहक से पूछ लेंगे।"
        if in_stock is True and price is None:
            text += " इसका दाम कितना है?"  # the call goes on: no thanks yet
        else:
            text += " धन्यवाद!"
        out[key] = text
    return out


def question_of(file_name: str) -> int | None:
    """Which question a recording answers, from its name: "q2_anything.m4a" -> 2. None when the name doesn't say."""
    found = re.match(r"q([1-9])(?![0-9])", os.path.basename(file_name).lower())
    n = int(found.group(1)) if found else None
    return n if n is not None and n <= len(QUESTIONS) else None


def find_clips(folder: str, limit: int = 20) -> list[dict]:
    """The owner's recordings under folder (any depth), sorted by name: [{"file", "synthetic", "question"}]."""
    found = []
    for root, _dirs, files in os.walk(folder or "/nonexistent"):
        found += [os.path.join(root, f) for f in files if f.lower().endswith(AUDIO) and not f.startswith(".")]
    found.sort(key=lambda p: (os.path.basename(p).lower(), p))
    return [{"file": p, "synthetic": False, "question": question_of(p)} for p in found[:limit]]


def wrong_fields(answer, expected: dict, alternative_any=()) -> list[str]:
    """The fields the answer got wrong (all four when there is no answer). The alternative also counts as right when
    it holds any of alternative_any (the model may write a brand in Latin or in Devanagari)."""
    if not isinstance(answer, dict):
        return list(FIELDS)
    wrong = []
    for f in FIELDS:
        got, want = answer.get(f), expected[f]
        named = f == "alternative" and isinstance(got, str) and isinstance(want, str)
        if named and any(w.lower() in got.lower() for w in alternative_any):
            continue
        if not same(f, got, want):
            wrong.append(f)
    return wrong


def new_results(asr_model: str, product: str) -> dict:
    """What voice_results.json starts as (the notebook fills it in, stage by stage)."""
    return {"asr_model": asr_model, "asr_licence": ASR_LICENCES.get(asr_model, "unknown: check before use"),
            "asr_device": None, "tts_model": None, "tts_licence": None, "tts_business_use": None, "tts_note": None,
            "tts_tried": [], "tts_device": None, "brain": BRAIN_INFO, "brain_processor": None, "product": product,
            "questions": [], "clips": [], "summary": [], "average_seconds": None, "errors": []}


def use_tts(results: dict, model: str) -> None:
    """Record which TTS model is used, with its licence (and that it is not for business use, when so)."""
    info = TTS_INFO.get(model, {"licence": "unknown: check before any use", "business_use": False,
                                "note": "not in the list of checked models"})
    results.update(tts_model=model, tts_licence=info["licence"], tts_business_use=info["business_use"],
                   tts_note=info["note"])


def tts_plan(candidates: list[str], token: str | None) -> tuple[list[str], list[dict]]:
    """Which TTS models to try, in order, and which are skipped and why (a model that needs a token, without one)."""
    plan, skipped = [], []
    for model in candidates:
        if TTS_INFO.get(model, {}).get("needs_token") and not token:
            skipped.append({"model": model, "ok": False, "why": "skipped: no HF_TOKEN (Kaggle Secrets or env)"})
        else:
            plan.append(model)
    return plan, skipped


def hf_token() -> str | None:
    """The Hugging Face token: Kaggle Secrets "HF_TOKEN" (Add-ons → Secrets), else the env HF_TOKEN. Never printed."""
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:  # noqa: BLE001 - not on Kaggle, or the secret is not attached to this notebook
        token = None
    return (token or os.environ.get("HF_TOKEN") or "").strip() or None


def timed(fn, *args):
    """fn(*args) and the seconds it took."""
    started = time.perf_counter()
    value = fn(*args)
    return value, round(time.perf_counter() - started, 2)


def process_clip(n: int, clip: dict, product: str, spoken: str | None, transcribe, think, speak, out_dir: str,
                 results: dict, save) -> dict:
    """One answer clip through the cascade: ears (ASR) -> brain (shop model) -> reply text -> mouth (TTS).

    The entry is added to results["clips"] first and save() runs after every stage, so a failure or a stop part-way
    loses nothing. A stage that fails is recorded (in the entry and in results["errors"]) and the next one still runs
    when it can: with no transcript or no answer, the reply is the polite request to say it again.
    """
    entry = {"n": n, "file": clip["file"], "synthetic": clip.get("synthetic", False), "question": clip.get("question"),
             "said": clip.get("said"), "expected": clip.get("expected"), "transcript": None, "transcript_auto": None,
             "reply": None, "answer": None, "wrong": None, "reply_text": None, "reply_speech": None,
             "reply_audio": None, "seconds": {"asr": None, "asr_auto": None, "brain": None, "tts": None, "total": None},
             "errors": []}
    results["clips"].append(entry)

    def stage(name, fn, *args):
        try:
            value, entry["seconds"][name] = timed(fn, *args)
            return value
        except Exception as e:  # noqa: BLE001 - record it, go on
            message = f"{name}: {e!r}"[:500]
            entry["errors"].append(message)
            results["errors"].append(f"clip {n} ({os.path.basename(clip['file'])}) {message}")
            return None
        finally:
            save()

    transcript = stage("asr", transcribe, clip["file"], "hi")
    entry["transcript"] = transcript.strip() if isinstance(transcript, str) else None
    auto = stage("asr_auto", transcribe, clip["file"], None)  # the language Whisper guesses (not in the total)
    entry["transcript_auto"] = auto.strip() if isinstance(auto, str) else None
    if entry["transcript"]:
        reply = stage("brain", think, product, entry["transcript"])
        entry["reply"] = reply.strip() if isinstance(reply, str) else None
        entry["answer"] = parse(entry["reply"]) if entry["reply"] else None
    if entry["expected"] is not None:
        entry["wrong"] = wrong_fields(entry["answer"], entry["expected"], clip.get("alternative_any") or ())
    reply = write_reply(entry["answer"], product, spoken)
    entry["reply_text"], entry["reply_speech"] = reply["text"], reply["speech"]
    path = os.path.join(out_dir, f"reply_{n}.wav")
    stage("tts", speak, reply["speech"], path, "ai")
    if entry["seconds"]["tts"] is not None:
        entry["reply_audio"] = path
    steps = [entry["seconds"][k] for k in ("asr", "brain", "tts")]
    entry["seconds"]["total"] = round(sum(s for s in steps if s is not None), 2)
    save()
    return entry


def average_seconds(clips: list[dict]) -> dict:
    """The average seconds per step over the clips where that step ran (None: it never ran)."""
    out = {}
    for step in ("asr", "brain", "tts", "total"):
        values = [c["seconds"][step] for c in clips if c.get("seconds", {}).get(step) is not None]
        out[step] = round(sum(values) / len(values), 2) if values else None
    return out


def short(text, width: int = 60) -> str:
    text = " ".join(str(text).split())
    return text if len(text) <= width else text[: width - 1] + "…"


def clip_line(c: dict) -> str:
    """One clip in one line: where it came from, what was heard, understood and replied, how right, how fast."""
    kind = "synthetic" if c.get("synthetic") else "your clip " + os.path.basename(c.get("file") or "?")
    question = f"q{c['question']}" if c.get("question") else "q?"
    if c.get("wrong") is None:
        mark = ""
    else:
        mark = " 4/4 right" if not c["wrong"] else f" {4 - len(c['wrong'])}/4 right (wrong: {', '.join(c['wrong'])})"
    s = c.get("seconds") or {}
    times = ", ".join(f"{k} {s[k]} s" for k in ("asr", "brain", "tts", "total") if s.get(k) is not None)
    heard = short(c["transcript"], 50) if c.get("transcript") else "(nothing heard)"
    return (f"{c.get('n', '?'):>2}. {kind} {question}{mark}: heard \"{heard}\" → {in_words(c.get('answer'))} → "
            f"reply \"{short(c.get('reply_text') or '(none)', 70)}\"" + (f" ({times})" if times else "")
            + (f" ERRORS: {'; '.join(c['errors'])}" if c.get("errors") else ""))


def summary_lines(results: dict) -> list[str]:
    """The run for a person: the models and licences, one line per clip, the synthetic score, the average times."""
    lines = [f"ASR (ears): {results.get('asr_model')} ({results.get('asr_licence')}) on {results.get('asr_device') or '?'}"]
    if results.get("tts_model"):
        flag = "" if results.get("tts_business_use") else " - NOT FOR BUSINESS USE: allowed for this internal test only"
        lines.append(f"TTS (mouth): {results['tts_model']} ({results.get('tts_licence')}){flag}")
    else:
        lines.append("TTS (mouth): none worked")
    for t in results.get("tts_tried") or []:
        if not t.get("ok"):
            lines.append(f"   TTS not used: {t.get('model')}: {short(t.get('why'), 200)}")
    lines.append(f"Brain: {results.get('brain')}; ran on {results.get('brain_processor') or '?'}")
    clips = results.get("clips") or []
    lines += [clip_line(c) for c in clips]
    synthetic = [c for c in clips if c.get("wrong") is not None]
    if synthetic:
        right = sum(not c["wrong"] for c in synthetic)
        lines.append(f"Synthetic answers: {right} of {len(synthetic)} with all 4 fields right "
                     "(a plumbing check with the TTS voice, not a real accent test)")
    avg = average_seconds(clips)
    if clips:
        lines.append(f"Average seconds over {len(clips)} clips: " + ", ".join(
            f"{k} {'?' if v is None else v}" for k, v in avg.items()))
    return lines


# The repo's Modelfile, to check the one made here is the same
MODELFILE_REPO = 'FROM ./wimb-shop-model.Q4_K_M.gguf\nTEMPLATE """{{ if .System }}<|im_start|>system\n{{ .System }}<|im_end|>\n{{ end }}{{ if .Prompt }}<|im_start|>user\n{{ .Prompt }}<|im_end|>\n{{ end }}<|im_start|>assistant\n{{ .Response }}<|im_end|>\n"""\nPARAMETER stop "<|im_end|>"\nPARAMETER temperature 0\nPARAMETER num_ctx 2048\nSYSTEM """You read what a shopkeeper said on a phone call about a product. Reply with only a JSON object with these keys: in_stock (true, false or null), price_inr (a number or null), delivery (true, false or null), alternative (text or null). price_inr is the price of the product that was asked for. Use null when the shopkeeper did not say."""\n'

RESULTS = new_results(ASR_MODEL, PRODUCT)
BRAIN_ERROR = None  # why the shop model is not running (None: it is, or not tried yet)
if os.path.isdir(os.path.dirname(WORKING_DIR)):
    os.makedirs(WORKING_DIR, exist_ok=True)
OLLAMA_LOG = "/tmp/ollama.log"  # what `ollama serve` prints (cell 7)


def save_results():
    """Write RESULTS to voice_results.json. Done after every stage, so a stop part-way loses nothing."""
    if os.path.isdir(WORKING_DIR):
        with open(os.path.join(WORKING_DIR, "voice_results.json"), "w", encoding="utf-8") as fh:
            json.dump(RESULTS, fh, ensure_ascii=False, indent=1)


def failed(stage, error):
    """Record a failed stage and go on (Kaggle's Run all stops at the first error that is not caught)."""
    message = f"{stage}: {error!r}" if isinstance(error, Exception) else f"{stage}: {error}"
    print("FAILED", message)
    RESULTS["errors"].append(message)
    save_results()


def sh(command, timeout=120):
    """What a shell command printed, or "" when the command is missing or fails."""
    try:
        return subprocess.run(command, shell=True, capture_output=True, text=True, timeout=timeout).stdout.strip()
    except Exception:
        return ""


def generate(body):
    request = urllib.request.Request(OLLAMA_URL + "/api/generate", json.dumps(body).encode(),
                                     {"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=600) as resp:
        return json.load(resp)


def in_memory():
    """The model as Ollama holds it now (from /api/ps: size, size_vram, ...), or None when it is not loaded."""
    with urllib.request.urlopen(OLLAMA_URL + "/api/ps", timeout=10) as resp:
        models = json.load(resp).get("models") or []
    return next((m for m in models if m.get("name", "").split(":")[0] == "wimb-shop"), None)


def processor():
    """Where the model runs now, as `ollama ps` says it: "100% GPU", "100% CPU", "25%/75% CPU/GPU", or None."""
    try:
        m = in_memory()
    except Exception:
        return None
    if not m or not m.get("size"):
        return None
    cpu = round(100 * (m["size"] - (m.get("size_vram") or 0)) / m["size"])
    return "100% GPU" if cpu <= 0 else "100% CPU" if cpu >= 100 else f"{cpu}%/{100 - cpu}% CPU/GPU"

In [ ]:
#@title 3. Install the speech tools (about 1–3 minutes)
# parler-tts is installed only when its model will be tried (an HF token is there). It pins transformers 4.46.1,
# so it goes in before anything imports transformers.
PARLER_PIP = "parler-tts==0.2.3"
HF_TOKEN = hf_token()  # Kaggle Secrets "HF_TOKEN", else the env HF_TOKEN (never printed)
TTS_PLAN, RESULTS["tts_tried"] = tts_plan(TTS_CANDIDATES, HF_TOKEN)
print("Hugging Face token:", "found" if HF_TOKEN else "none (Indic Parler-TTS is skipped)")


def pip(*packages):
    done = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages],
                          capture_output=True, text=True, timeout=1800)
    if done.returncode != 0:
        raise RuntimeError((done.stdout + done.stderr)[-1500:])


try:
    pip("soundfile", "librosa")  # usually there already
except Exception as e:
    failed("pip install soundfile librosa", e)
if not shutil.which("ffmpeg"):  # ffmpeg reads .m4a, .ogg and .opus (phone recordings)
    sh("apt-get update -qq; DEBIAN_FRONTEND=noninteractive apt-get install -y -qq ffmpeg", timeout=900)
RESULTS["ffmpeg"] = bool(shutil.which("ffmpeg"))
if any("parler" in m for m in TTS_PLAN):
    try:
        pip(PARLER_PIP)
    except Exception as e:
        failed("pip install " + PARLER_PIP, e)
        RESULTS["tts_tried"] += [{"model": m, "ok": False, "why": "parler-tts did not install"}
                                 for m in TTS_PLAN if "parler" in m]
        TTS_PLAN = [m for m in TTS_PLAN if "parler" not in m]
try:
    import transformers  # noqa: F401 - only to see that it is there
except ImportError:
    pip("transformers", "accelerate")
import importlib.metadata as md


def version(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return None


RESULTS["versions"] = {p: version(p) for p in ("torch", "transformers", "parler_tts", "soundfile", "librosa")}
print("ffmpeg:", RESULTS["ffmpeg"], "| versions:", RESULTS["versions"], "| TTS to try:", TTS_PLAN)
save_results()

In [ ]:
#@title 4. Ears: load Whisper (speech to text)
ASR = None  # the speech-to-text pipeline, once loaded


def load_audio(path):
    """16 kHz mono samples. ffmpeg reads any format (.m4a, .ogg, .opus too); without it, librosa."""
    import tempfile

    import numpy as np
    import soundfile as sf

    if shutil.which("ffmpeg"):
        with tempfile.TemporaryDirectory() as tmp:
            wav = os.path.join(tmp, "audio.wav")
            done = subprocess.run(["ffmpeg", "-nostdin", "-loglevel", "error", "-y", "-i", path, "-ac", "1",
                                   "-ar", "16000", wav], capture_output=True, text=True, timeout=300)
            if done.returncode != 0:
                raise RuntimeError(f"ffmpeg could not read {os.path.basename(path)}: {done.stderr[-300:]}")
            audio, _ = sf.read(wav, dtype="float32")
        return audio
    import librosa

    audio, _ = librosa.load(path, sr=16000, mono=True)
    return audio.astype(np.float32)


def transcribe(path, language):
    """What was said in the recording: language "hi" (Hindi), or None to let Whisper guess the language."""
    if ASR is None:
        raise RuntimeError("the speech-to-text model did not load (see cell 4)")
    options = {"task": "transcribe", **({"language": language} if language else {})}
    return ASR({"raw": load_audio(path), "sampling_rate": 16000}, generate_kwargs=options)["text"].strip()


try:
    import torch
    from transformers import pipeline

    on_gpu = torch.cuda.is_available()
    started = time.time()
    ASR = pipeline("automatic-speech-recognition", model=ASR_MODEL, device="cuda:0" if on_gpu else "cpu",
                   torch_dtype=torch.float16 if on_gpu else torch.float32, chunk_length_s=30)
    RESULTS["asr_device"] = f"GPU ({torch.cuda.get_device_name(0)})" if on_gpu else "CPU"
    RESULTS["asr_load_seconds"] = round(time.time() - started, 1)
    print(f"{ASR_MODEL} loaded in {RESULTS['asr_load_seconds']} s on the {RESULTS['asr_device']}")
except Exception as e:
    failed("load the speech-to-text model", e)
save_results()

In [ ]:
#@title 5. Mouth: load the text to speech (the first that works)
TTS = None  # {"model", "say": say(text, voice) -> (samples, sample rate)}, once one loads
# Indic Parler-TTS takes the voice as a description (Divya and Rohit are two of its Hindi speakers).
VOICES = {"ai": "Divya speaks in a clear, calm and polite voice at a moderate pace. The recording is very clear, "
                "with no background noise.",
          "shopkeeper": "Rohit speaks quickly in a casual tone. The recording is close, with a little background noise."}


def device_name():
    import torch

    return "cuda:0" if torch.cuda.is_available() else "cpu"


def load_parler(model):
    """Indic Parler-TTS (needs the HF token and parler-tts). Two voices: the AI's and the shopkeeper's."""
    import torch
    from parler_tts import ParlerTTSForConditionalGeneration
    from transformers import AutoTokenizer

    device = device_name()
    net = ParlerTTSForConditionalGeneration.from_pretrained(model, token=HF_TOKEN).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model, token=HF_TOKEN)
    describe = AutoTokenizer.from_pretrained(net.config.text_encoder._name_or_path)

    def say(text, voice):
        d = describe(VOICES[voice], return_tensors="pt").to(device)
        p = tokenizer(text, return_tensors="pt").to(device)
        with torch.no_grad():
            audio = net.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                                 prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)
        return audio.cpu().float().numpy().squeeze(), net.config.sampling_rate

    return say, device


def load_mms(model):
    """Meta's MMS-TTS (VITS, no token needed). One voice for both sides."""
    import torch
    from transformers import AutoTokenizer, VitsModel

    device = device_name()
    net = VitsModel.from_pretrained(model).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model)
    romanize = None
    if getattr(tokenizer, "is_uroman", False):  # this voice reads Latin letters only: romanize the text first
        pip("uroman")
        import uroman

        romanize = uroman.Uroman().romanize_string

    def say(text, voice):
        inputs = tokenizer(romanize(text) if romanize else text, return_tensors="pt").to(device)
        torch.manual_seed(0)  # the same audio on every run
        with torch.no_grad():
            audio = net(**inputs).waveform
        return audio.cpu().float().numpy().squeeze(), net.config.sampling_rate

    return say, device


def speak(text, path, voice="ai"):
    """Say the text into a .wav file. voice: "ai" or "shopkeeper"."""
    import soundfile as sf

    if TTS is None:
        raise RuntimeError("no text-to-speech model loaded (see cell 5)")
    audio, rate = TTS["say"](text, voice)
    if audio.size == 0:
        raise RuntimeError("the text-to-speech model made no sound")
    sf.write(path, audio, rate)


for model in TTS_PLAN:
    started = time.time()
    try:
        say, device = (load_parler if "parler" in model else load_mms)(model)
        say("नमस्ते", "ai")  # a first try: some problems only show here
    except Exception as e:  # e.g. gated: the token's account has not accepted the model's terms
        RESULTS["tts_tried"].append({"model": model, "ok": False, "why": f"{e!r}"[:500]})
        print(f"{model} did not work:", f"{e!r}"[:600])
        continue
    TTS = {"model": model, "say": say}
    use_tts(RESULTS, model)
    RESULTS["tts_device"] = device
    RESULTS["tts_tried"].append({"model": model, "ok": True, "why": f"loaded in {round(time.time() - started, 1)} s"})
    break
if TTS is None:
    failed("load a text-to-speech model", "none of " + ", ".join(TTS_CANDIDATES) + " worked")
else:
    print(f"Text to speech: {RESULTS['tts_model']} ({RESULTS['tts_licence']}) on {RESULTS['tts_device']}")
    if not RESULTS["tts_business_use"]:
        print("NOT FOR BUSINESS USE:", RESULTS["tts_note"])
save_results()

In [ ]:
#@title 6. Brain: find the trained model (the training notebook's output, attached as input)
if BRAIN_ERROR is None:  # skipped when an earlier brain step failed
    try:
        import glob


        def rank(path):
            name = os.path.basename(path).lower()
            return ("wimb-shop-model" not in name or "q4_k_m" not in name, "q4_k_m" not in name, path)


        found = sorted(glob.glob("/kaggle/input/**/*.gguf", recursive=True), key=rank)
        if not found:
            print("No .gguf file under /kaggle/input. What is attached:")
            for root, dirs, files in os.walk("/kaggle/input"):
                depth = root[len("/kaggle/input"):].count(os.sep)
                print("   " * depth + (os.path.basename(root) or root) + "/")
                for name in sorted(files)[:20]:
                    print("   " * (depth + 1) + name)
                if depth >= 3:
                    dirs[:] = []
            failed("find the model", "no .gguf file under /kaggle/input")
            raise RuntimeError("No model file. Attach the training notebook's output: Add Input → Notebooks → Your Work → "
                               "wimb-shop-model-training, then Run all again.")
        MODEL_FILE = found[0]
        RESULTS["model_file"] = MODEL_FILE
        RESULTS["model_mb"] = round(os.path.getsize(MODEL_FILE) / 1e6)
        print(f"Model file: {MODEL_FILE}  ({RESULTS['model_mb']:,} MB)")
        save_results()
    except Exception as e:
        BRAIN_ERROR = f"{type(e).__name__}: {e}"[:500]
        failed("brain", e)
        print("The brain is NOT running (the ears and the mouth still run):", BRAIN_ERROR)
save_results()

In [ ]:
#@title 7. Brain: install Ollama and start it (about 1–2 minutes)
if BRAIN_ERROR is None:  # skipped when an earlier brain step failed
    try:
        OLLAMA_URL = "http://127.0.0.1:11434"


        def ollama_version():
            try:
                with urllib.request.urlopen(OLLAMA_URL + "/api/version", timeout=5) as resp:
                    return json.load(resp).get("version")
            except Exception:
                return None


        if not ollama_version():
            if not shutil.which("ollama"):
                # zstd unpacks Ollama's download (.tar.zst). The download already holds the CUDA libraries; lspci is
                # left out on purpose: with it (and no nvidia-smi) install.sh would try to install NVIDIA drivers.
                if not shutil.which("zstd"):
                    sh("apt-get update -qq; DEBIAN_FRONTEND=noninteractive apt-get install -y -qq zstd", timeout=600)
                install = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True,
                                         capture_output=True, text=True, timeout=1200)
                print("\n".join((install.stdout + install.stderr).replace("\r", "\n").splitlines()[-15:]))
                if not shutil.which("ollama"):
                    failed("install Ollama", (install.stdout + install.stderr)[-1500:])
                    raise RuntimeError("Ollama did not install (see above)")
            print(sh("ollama --version 2>&1"))
            log = open(OLLAMA_LOG, "w")  # stays open: the server writes to it
            server = subprocess.Popen(["ollama", "serve"], env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"},
                                      stdout=log, stderr=subprocess.STDOUT)
            for _ in range(60):
                if ollama_version() or server.poll() is not None:
                    break
                time.sleep(1)
        if not ollama_version():
            tail = sh(f"tail -n 40 {OLLAMA_LOG}")
            print(tail)
            failed("start Ollama", tail[-1500:])
            raise RuntimeError("Ollama did not start (its log is above)")
        RESULTS["ollama_version"] = ollama_version()
        RESULTS["gpu"] = (sh("nvidia-smi --query-gpu=name --format=csv,noheader").splitlines() or [None])[0]
        print("Ollama", RESULTS["ollama_version"], "is running. GPU:", RESULTS["gpu"])
        save_results()
    except Exception as e:
        BRAIN_ERROR = f"{type(e).__name__}: {e}"[:500]
        failed("brain", e)
        print("The brain is NOT running (the ears and the mouth still run):", BRAIN_ERROR)
save_results()

In [ ]:
#@title 8. Brain: make the Ollama model from the file, exactly as on a PC (ollama create, about 1 minute)
if BRAIN_ERROR is None:  # skipped when an earlier brain step failed
    try:
        os.makedirs("/tmp/wimb", exist_ok=True)
        link = "/tmp/wimb/wimb-shop-model.Q4_K_M.gguf"
        if os.path.lexists(link):
            os.remove(link)
        os.symlink(MODEL_FILE, link)
        modelfile = make_modelfile("wimb-shop-model.Q4_K_M.gguf")
        if modelfile != MODELFILE_REPO:
            failed("Modelfile", "the Modelfile made here differs from the repo's Modelfile")
            raise RuntimeError("The Modelfile made here differs from the repo's Modelfile: run build_notebook.py again")
        with open("/tmp/wimb/Modelfile", "w", encoding="utf-8") as fh:
            fh.write(modelfile)
        print(modelfile)


        def create():
            started = time.time()
            done = subprocess.run(["ollama", "create", "wimb-shop", "-f", "/tmp/wimb/Modelfile"],
                                  capture_output=True, text=True, timeout=1800)
            output = re.sub(r"\x1b\[[0-9;?]*[A-Za-z]", "", done.stdout + done.stderr).replace("\r", "\n")  # no progress codes
            print("\n".join(line for line in output.splitlines()[-8:] if line.strip()))
            return done.returncode, output, round(time.time() - started, 1)


        exit_code, output, RESULTS["create_seconds"] = create()
        if exit_code != 0:  # some versions do not follow the link: try once with a real copy
            print("ollama create failed with the linked file; trying again with a copy")
            os.remove(link)
            shutil.copy(MODEL_FILE, link)
            exit_code, output, RESULTS["create_seconds"] = create()
        if exit_code != 0:
            failed("ollama create", output[-1500:])
            raise RuntimeError("ollama create failed (see above)")
        print(f"ollama create took {RESULTS['create_seconds']} s")
        save_results()

        started = time.time()
        generate({"model": "wimb-shop", "keep_alive": "60m"})  # no prompt: only load it
        RESULTS["brain_load_seconds"] = round(time.time() - started, 1)
        print(f"Shop model loaded in {RESULTS['brain_load_seconds']} s")
    except Exception as e:
        BRAIN_ERROR = f"{type(e).__name__}: {e}"[:500]
        failed("brain", e)
        print("The brain is NOT running (the ears and the mouth still run):", BRAIN_ERROR)
save_results()


def think(product, transcript):
    """The shop model's reply (the 4-field JSON) to what the shopkeeper said, through Ollama's API."""
    if BRAIN_ERROR:
        raise RuntimeError(f"the shop model is not running ({BRAIN_ERROR})")
    data = generate({"model": "wimb-shop", "prompt": user_message(product, transcript), "stream": False,
                     "keep_alive": "60m", "options": {"temperature": 0, "num_predict": 96}})
    return data.get("response", "").strip()


print("The brain is", "ready" if BRAIN_ERROR is None else f"NOT running: {BRAIN_ERROR} (ears and mouth still run)")

In [ ]:
#@title 9. The AI's 3 questions, spoken (q1.wav, q2.wav, q3.wav)
RESULTS["questions"] = []
for q in questions(PRODUCT, PRODUCT_SPOKEN):
    entry = {**q, "audio": None, "seconds": None}
    path = os.path.join(WORKING_DIR, f"q{q['n']}.wav")
    try:
        _, entry["seconds"] = timed(speak, q["speech"], path, "ai")
        entry["audio"] = path
    except Exception as e:
        failed(f"question {q['n']}", e)
    RESULTS["questions"].append(entry)
    print(f"q{q['n']}: {q['text']}  →  {entry['audio'] or 'no audio'}")
save_results()

In [ ]:
#@title 10. The answers: your recordings (if attached) and 6 synthetic ones
CLIPS = find_clips(CLIPS_DIR, MAX_CLIPS)
print(f"Your recordings: {len(CLIPS)}" + ("" if CLIPS else " (none attached: only the synthetic answers)"))
for c in CLIPS:
    print("   ", c["file"], "answers", f"q{c['question']}" if c["question"] else "? (the name doesn't say)")
made = 0
for i, row in enumerate(SYNTHETIC, 1):  # made-up answers, said by the TTS: a plumbing check, not an accent test
    path = os.path.join(WORKING_DIR, f"synthetic_{i}.wav")
    try:
        speak(row["said"], path, "shopkeeper")
    except Exception as e:
        failed(f"synthetic answer {i}", e)
        continue
    made += 1
    CLIPS.append({"file": path, "synthetic": True, "question": row["question"], "said": row["said"],
                  "expected": row["answer"], "alternative_any": row.get("alternative_any", [])})
print(f"Synthetic answers spoken: {made} of {len(SYNTHETIC)}")
RESULTS["clips_planned"] = len(CLIPS)
save_results()

In [ ]:
#@title 11. Each answer: ears → brain → reply → mouth (timed)
RESULTS["clips"] = []
for n, clip in enumerate(CLIPS, 1):
    entry = process_clip(n, clip, PRODUCT, PRODUCT_SPOKEN, transcribe, think, speak, WORKING_DIR, RESULTS,
                         save_results)
    print(clip_line(entry))
save_results()

In [ ]:
#@title 12. Summary (saved to voice/voice_results.json)
if BRAIN_ERROR is None:
    RESULTS["brain_processor"] = processor()  # where Ollama ran the shop model: "100% GPU", ...
RESULTS["average_seconds"] = average_seconds(RESULTS["clips"])
RESULTS["summary"] = summary_lines(RESULTS)
print("\n".join(RESULTS["summary"]))
print("\nErrors:", "none" if not RESULTS["errors"] else "")
for error in RESULTS["errors"]:
    print("   ", error[:500])
save_results()
print("\nListen: the Output tab → voice/ (q1-q3.wav: the questions; synthetic_N.wav: the test answers; "
      "reply_N.wav: the replies)")

## What next
- **Your own voice:** record answers on your phone (`q1_….m4a`, `q2_….m4a`, `q3_….m4a`), put them in a private
  Kaggle dataset and run again with it attached (README: "Voice v0").
- This cascade (ears → brain → mouth) is the start. The goal is a direct speech-to-speech model, trained on real
  calls recorded with the shopkeepers' consent.